# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library. You'll learn how to load metadata, discover record sets, extract tabular data, perform exploratory analysis, and visualize key aspects using only the dataset's Croissant schema.

### Dataset Source
The dataset is available via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

- Licensed under: [Open Data Commons Attribution License 1.0](https://opendatacommons.org/licenses/by/1-0/)
- Identifier: `10.71728/senscience.y7m0-f273`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"License: {getattr(metadata, 'license', None)}")


## 2. Data Overview
Let's inspect the available record sets, their IDs, and their fields, including column and field `@id`s. This will let us reference all entities by their unique `@id` going forward.

In [ ]:
# List and inspect all record sets and their fields in the dataset using their @id

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.\nPlease check the dataset structure and schema.")
else:
    print(f"{len(record_sets)} record set(s) found:\n")
    for rs in record_sets:
        print(f"- RecordSet name: {rs.name}")
        print(f"  @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {getattr(f, 'name', 'Unknown')} (Field @id: {f.id})")
                # Try to list columns within this field (if they exist)
                if hasattr(f, 'columns') and f.columns:
                    print(f"      Columns:")
                    for col in f.columns:
                        print(f"        - {getattr(col, 'name', 'Unknown')} (@id: {col.id})")
        print()


## 3. Data Extraction
Extract data from each record set into pandas DataFrames for analysis. All record sets and fields are referenced using their respective `@id`.

> **Note**: If your dataset contains only one main record set, use its `@id` as referenced above. If multiple, select one for demonstration.

In [ ]:
# Extract all data into DataFrames, using record set @id

dataframes = {}
all_record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")

# For demonstration, preview the first available record set's DataFrame
if all_record_set_ids:
    example_id = all_record_set_ids[0]
    print(f"Preview of DataFrame for RecordSet @id: {example_id}")
    display(dataframes[example_id].head())


## 4. Exploratory Data Analysis (EDA)
Let's perform simple numeric filtering, normalization, and grouping by a categorical field on the extracted data.

- Pick a numeric field and a grouping field from the DataFrame columns shown in the previous section.
- All field and column names are referenced by their `@id`.


In [ ]:
import numpy as np

# Use the first available DataFrame for illustration
if all_record_set_ids:
    record_set_id = all_record_set_ids[0]
    df = dataframes[record_set_id]

    # List columns so user can pick fields by @id
    print(f"Available columns (@id) in RecordSet {record_set_id}:")
    print(df.columns.tolist())

    # Example: select the first numeric column found for illustration
    # You can customize this section as per your field structure
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered {record_set_id} records where {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize the numeric column
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized column {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical/grouping field (non-numeric)
        group_field_id = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No suitable group field (categorical) found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No record sets loaded; cannot perform analysis.")

## 5. Visualization

Let's visualize the numeric field (from the previous section) and its distribution after normalization. If a grouping field was found, we'll also plot grouped statistics.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_record_set_ids and numeric_field_id:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field_id], ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field_id}")

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], ax=axs[1], kde=True)
        axs[1].set_title(f"Normalized {numeric_field_id} (filtered)")

    plt.tight_layout()
    plt.show()

    # Barplot if grouping field is found
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped.index.astype(str), y=grouped.values)
        plt.xticks(rotation=45)
        plt.title(f"Grouped Mean of {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion

In this notebook, you've explored the FAIR² dataset's structure and content using the Croissant schema and the `mlcroissant` library. Starting from the Croissant metadata, you:
- Discovered all available record sets and their associated fields and IDs.
- Loaded tabular data using canonical `@id` references for all entities.
- Performed numeric filtering, normalization, and grouping analyses.
- Visualized the core numeric field distribution and group-wise aggregations.

To continue: Explore other record sets or columns using their `@id`, or apply advanced ML methods to the extracted data!